In [ ]:
import csv
import hashlib
import json
import platform
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# ==================================================
# 1. PROVENANCE ET CONFIGURATION
# ==================================================

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
GIT_REF = "exp/cae-bottle-comparison-st030602-v1"
NOTEBOOK_PATH = "notebooks/11-cae-bottle-bottleneck-comparison.ipynb"

STUDY_ID = "US-03-06-CMP-001"
DATASET_VERSION = "2"
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_root = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
output_dir = Path("/kaggle/working/us-03-06-comparison")
output_dir.mkdir(parents=True, exist_ok=True)

assert (dataset_root / "bottle/train/good").is_dir()

repo = Path(tempfile.mkdtemp(prefix="industrial-defect-ai-")) / "repository"

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch",
     GIT_REF, REPO_URL, str(repo)],
    check=True,
)

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip()

assert (repo / NOTEBOOK_PATH).is_file()
sys.path.insert(0, str(repo))


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# ==================================================
# 2. CHARGER LES CHECKPOINTS EXP-002 / EXP-003
# ==================================================

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.inference.cae_inference import (
    CAEInference,
    CAEInferenceConfig,
)

experiments = {
    "EXP-002": {
        "channels": 128,
        "config": "experiments/configs/cae_baseline.json",
    },
    "EXP-003": {
        "channels": 32,
        "config": "experiments/configs/cae_bottle_bottleneck32.json",
    },
}

checkpoint_candidates = list(
    Path("/kaggle/input").rglob("bottle_best.pth")
)

engines = {}
metadata = {}
checkpoint_paths = {}

inference_config = CAEInferenceConfig(
    smoothing_sigma=4.0,
    score_method="max",
)

for exp_id, settings in experiments.items():

    meta = read_json(
        repo / "experiments/runs/cae-bottle" / f"{exp_id}.json"
    )
    metadata[exp_id] = meta

    expected_sha = meta["artifacts"]["model_sha256"]

    matches = [
        p for p in checkpoint_candidates
        if sha256(p) == expected_sha
    ]

    if not matches:
        raise FileNotFoundError(
            f"Checkpoint {exp_id} introuvable dans Kaggle Input."
        )

    checkpoint_path = sorted(matches)[0]
    checkpoint_paths[exp_id] = checkpoint_path

    engine = CAEInference.from_checkpoint(
        category="bottle",
        config_path=repo / settings["config"],
        checkpoint_path=checkpoint_path,
        device=device,
        config=inference_config,
    )

    assert engine.model.config.latent_channels == settings["channels"]
    assert meta["training"]["architecture"]["latent_channels"] == settings["channels"]

    engines[exp_id] = engine

    print(
        exp_id,
        "| bottleneck:", settings["channels"],
        "| checkpoint SHA256: OK"
    )

# ==================================================
# 3. CHARGER LA VALIDATION
# ==================================================

manifest_path = repo / "docs/data/us-02-03/bottle_manifest.csv"
split_protocol = read_json(
    repo / "docs/data/us-02-03/split_protocol.json"
)

assert sha256(manifest_path) == (
    split_protocol["manifests"]["bottle"]["sha256"]
)

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest_path,
    category="bottle",
    split="val",
    transform=MVTecTransform(
        size=(256, 256),
        profile="cae",
    ),
)

assert len(val_dataset) == 42
assert all(s["label"] == 0 for s in val_dataset.samples)

# Charger une seule fois pour les deux modèles.
images = [val_dataset[i]["image"] for i in range(42)]


# ==================================================
# 4. GÉNÉRER LES MÊMES DÉFAUTS SYNTHÉTIQUES
# ==================================================

def add_synthetic_defect(image, index):
    """Ajoute un carré de bruit 32x32, déterministe."""
    rng = np.random.default_rng(SEED + index)
    result = image.clone()

    size = 32
    y = int(rng.integers(16, 256 - size - 16))
    x = int(rng.integers(16, 256 - size - 16))

    noise = rng.random((3, size, size)).astype(np.float32)
    result[:, y:y + size, x:x + size] = torch.from_numpy(noise)

    return result


synthetic_images = [
    add_synthetic_defect(image, i)
    for i, image in enumerate(images)
]

# ==================================================
# 5. INFÉRENCE ET SCORES
# ==================================================

results = {}
predictions = {}

for exp_id, engine in engines.items():

    normal_predictions = [
        engine.predict(image)
        for image in images
    ]

    defect_predictions = [
        engine.predict(image)
        for image in synthetic_images
    ]

    predictions[exp_id] = defect_predictions

    normal_scores = np.array([
        p.score for p in normal_predictions
    ])

    defect_scores = np.array([
        p.score for p in defect_predictions
    ])

    val_mse = float(np.mean([
        p.extras["mean_mse"]
        for p in normal_predictions
    ]))

    reference_mse = metadata[exp_id]["metrics"]["best_val_mse"]

    assert np.isclose(
        val_mse, reference_mse, rtol=1e-3, atol=1e-6
    ), f"MSE restaurée incorrecte : {exp_id}"

    # AUC exploratoire : normales vs défauts SYNTHÉTIQUES.
    # Ce n'est PAS l'Image-AUROC officielle de MVTec AD.
    synthetic_auc = float(
        np.mean(defect_scores[:, None] > normal_scores[None, :])
        + 0.5 * np.mean(
            defect_scores[:, None] == normal_scores[None, :]
        )
    )

    results[exp_id] = {
        "latent_channels": experiments[exp_id]["channels"],
        "parameters": sum(
            p.numel() for p in engine.model.parameters()
        ),
        "checkpoint_mb": checkpoint_paths[exp_id].stat().st_size / 1e6,
        "validation_mse": val_mse,
        "mean_normal_score": float(normal_scores.mean()),
        "mean_synthetic_score": float(defect_scores.mean()),
        "median_score_ratio": float(np.median(
            defect_scores / np.maximum(normal_scores, 1e-12)
        )),
        "synthetic_score_increase_rate": float(
            np.mean(defect_scores > normal_scores)
        ),
        "synthetic_auroc_exploratory": synthetic_auc,
        "training_seconds": metadata[exp_id]["metrics"]["total_training_seconds"],
    }

    results[exp_id]["normal_scores"] = normal_scores.tolist()
    results[exp_id]["synthetic_scores"] = defect_scores.tolist()

    print(
        exp_id,
        "| Validation MSE:", round(val_mse, 8),
        "| Synthetic AUC:", round(synthetic_auc, 4)
    )

# ==================================================
# 6. MESURER LA LATENCE
# ==================================================

def benchmark(engine, image, warmup=10, repetitions=50):
    durations = []

    for _ in range(warmup):
        engine.predict(image)

    if device.type == "cuda":
        torch.cuda.synchronize()

    for _ in range(repetitions):
        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        engine.predict(image)

        if device.type == "cuda":
            torch.cuda.synchronize()

        durations.append(
            (time.perf_counter() - start) * 1000
        )

    return {
        "median_ms": float(np.median(durations)),
        "mean_ms": float(np.mean(durations)),
        "std_ms": float(np.std(durations)),
        "repetitions": repetitions,
    }


for exp_id, engine in engines.items():
    latency = benchmark(engine, images[0])
    results[exp_id]["latency"] = latency

    print(
        exp_id,
        "| Latence médiane:",
        round(latency["median_ms"], 3),
        "ms/image"
    )

# ==================================================
# 7. VISUALISER LES RECONSTRUCTIONS ET CARTES
# ==================================================

fig, axes = plt.subplots(3, 6, figsize=(18, 10))

for row, index in enumerate([0, 1, 2]):

    original = images[index].permute(1, 2, 0).numpy()
    synthetic = synthetic_images[index].permute(1, 2, 0).numpy()

    p128 = predictions["EXP-002"][index]
    p32 = predictions["EXP-003"][index]

    recon128 = p128.reconstruction.permute(1, 2, 0).numpy()
    recon32 = p32.reconstruction.permute(1, 2, 0).numpy()

    map128 = p128.anomaly_map.numpy()
    map32 = p32.anomaly_map.numpy()

    vmax = max(map128.max(), map32.max(), 1e-12)

    contents = [
        (original, "Original"),
        (synthetic, "Defaut synthetique"),
        (recon128, "Reconstruction 128"),
        (recon32, "Reconstruction 32"),
        (map128, "Carte 128"),
        (map32, "Carte 32"),
    ]

    for col, (data, title) in enumerate(contents):
        if col < 4:
            axes[row, col].imshow(data)
        else:
            axes[row, col].imshow(
                data, cmap="inferno", vmin=0, vmax=vmax
            )

        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(output_dir / "comparison_examples.png", dpi=150)
plt.show()
plt.close()

# Distribution des scores.
fig, ax = plt.subplots(figsize=(10, 5))

score_groups = [
    results["EXP-002"]["normal_scores"],
    results["EXP-002"]["synthetic_scores"],
    results["EXP-003"]["normal_scores"],
    results["EXP-003"]["synthetic_scores"],
]

ax.boxplot(score_groups, showfliers=False)
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels([
    "128 normal", "128 synthetique",
    "32 normal", "32 synthetique",
])
ax.set_ylabel("Score brut (max, sigma=4)")
ax.set_title("Bottle - comparaison des scores")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "comparison_scores.png", dpi=150)
plt.show()
plt.close()

# ==================================================
# 8. ENREGISTRER LE RAPPORT REPRODUCTIBLE
# ==================================================

report = {
    "study_id": STUDY_ID,
    "category": "bottle",
    "purpose": "bottleneck capacity comparison",
    "reproducibility": {
        "git_ref": GIT_REF,
        "git_commit": COMMIT,
        "notebook": NOTEBOOK_PATH,
        "notebook_sha256": sha256(repo / NOTEBOOK_PATH),
        "dataset_source": "ipythonx/mvtec-ad",
        "dataset_version": DATASET_VERSION,
        "split": "val",
        "split_manifest_sha256": sha256(manifest_path),
        "validation_images": len(images),
        "seed": SEED,
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "device": str(device),
        "gpu": (
            torch.cuda.get_device_name(0)
            if device.type == "cuda" else None
        ),
    },
    "protocol": {
        "score_method": "max",
        "smoothing_sigma": 4.0,
        "synthetic_defect": "32x32 uniform random noise",
        "same_images_for_both_models": True,
        "official_test_used": False,
        "threshold_calibration": False,
        "latency_includes_preprocessed_tensor_to_cpu_result": True,
    },
    "models": {
        exp_id: {
            "checkpoint_sha256": metadata[exp_id]["artifacts"]["model_sha256"],
            "training_git_commit": metadata[exp_id]["reproducibility"]["git_commit"],
            "training_kaggle_url": metadata[exp_id]["artifacts"]["kaggle_version_url"],
            **results[exp_id],
        }
        for exp_id in experiments
    },
    "limitations": (
        "Synthetic defects are used for exploratory comparison only. "
        "Results do not measure real-world anomaly detection performance."
    ),
}

report_path = output_dir / "comparison_report.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

with (output_dir / "pip-freeze.txt").open("w", encoding="utf-8") as f:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=f,
        check=True,
    )

print("\nST-03-06-02 — COMPARAISON TERMINÉE")
print("Git commit:", COMMIT)
print("GPU:", report["reproducibility"]["gpu"])

for exp_id, metrics in results.items():
    print(
        f"{exp_id}: "
        f"Val MSE={metrics['validation_mse']:.8f}, "
        f"Synthetic AUC={metrics['synthetic_auroc_exploratory']:.4f}, "
        f"Latency={metrics['latency']['median_ms']:.3f} ms"
    )

print("\nArtefacts :")
for file in sorted(output_dir.iterdir()):
    print(" -", file.name)